# Lab 3 — The Prompt Contract

**Module:** Instruction-Following & Alignment | **Duration:** 90 min | **Pair programming:** Switch at 45 min

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Objectives

1. Write prompts as engineering contracts (objective, constraints, format, evaluation criteria)
2. Iterate from naive → improved → hardened through systematic evaluation
3. Build a test suite that measures prompt reliability
4. Connect prompt engineering to alignment mechanisms (instruction tuning, RLHF)

**Core principle:** *Instruction tuning makes the model responsive to instructions, but a vague instruction still produces vague output. The prompt contract is how you engineer reliability.*

---
## Setup

In [ ]:
import json
import requests
import sys
import os

# Add project root to path
sys.path.insert(0, '.')

from utils.generation_utils import (
    generate, generate_n, is_ollama_available,
    load_precomputed, load_test_inputs, try_parse_json
)
from utils.test_runner import run_test_suite

print("✓ All imports successful!")

In [ ]:
# === Check Ollama ===
OLLAMA_AVAILABLE = is_ollama_available()
if OLLAMA_AVAILABLE:
    print("✓ Ollama is running.")
else:
    print("⚠ Ollama not reachable. Using pre-computed outputs.")

PRECOMPUTED = load_precomputed()
if PRECOMPUTED:
    print("✓ Pre-computed outputs loaded.")

In [ ]:
# === Load available tasks ===
with open("data/sample_tasks.json", "r") as f:
    all_tasks = json.load(f)

print("Available tasks:")
for task_id, task_data in all_tasks["tasks"].items():
    print(f"  • {task_id}: {task_data['description'][:80]}...")

---
## Step 1: Choose Your Task

Select one of the 6 tasks above. Set `TASK_ID` below.

In [ ]:
# TODO: Choose your task
TASK_ID = "ticket_summarizer"  # Change to your chosen task

task_info = all_tasks["tasks"][TASK_ID]
test_inputs = task_info["inputs"]

print(f"Task: {TASK_ID}")
print(f"Description: {task_info['description']}")
print(f"Available inputs: {len(test_inputs)} (5 normal + 1 edge + 1 adversarial)")

# Preview first input
first_key = list(test_inputs[0].keys())[0]
print(f"\nFirst input preview ({first_key}):")
print(f"  {str(test_inputs[0][first_key])[:200]}...")

---
# PART 1 — THREE PROMPT ITERATIONS (55 minutes)

**🟢 Driver starts here. Switch roles after V1.**

---

## Version 1: The Naive Prompt (15 min)

Write the simplest prompt you can think of. No formatting, no constraints — just ask.

In [ ]:
# ============================================================
# VERSION 1: NAIVE PROMPT
# ============================================================
# Write the simplest, most natural prompt for your task.
# Use {placeholder} for the input field.
#
# Examples:
#   ticket_summarizer:    "Summarize this support ticket: {ticket_text}"
#   acceptance_criteria:  "Write acceptance criteria for: {user_story}"
#   field_extractor:      "Extract key fields from this job posting: {job_posting}"
#   code_review:          "Review this code: {code}"
#   meeting_notes:        "Create meeting notes from: {transcript}"
#   email_rewriter:       "Rewrite this email professionally: {email_draft}"
# ============================================================

naive_prompt = """Summarize this support ticket: {ticket_text}"""
# TODO: Replace with your prompt for your chosen task

print(f"V1 Prompt:\n{naive_prompt}")

In [ ]:
# === Run V1 on 5 normal inputs ===
results_v1 = []

if OLLAMA_AVAILABLE:
    for i, inp in enumerate(test_inputs[:5]):
        prompt = naive_prompt.format(**inp)
        response = generate(prompt)
        results_v1.append({"input_id": i, "response": response})
        print(f"--- Input {i} ---")
        print(f"Response: {response[:400]}")
        print()
else:
    print("Using pre-computed V1 outputs:")
    v1_data = PRECOMPUTED.get(TASK_ID, {}).get("v1_naive", {})
    for i, output in enumerate(v1_data.get("outputs", [])[:5]):
        results_v1.append({"input_id": i, "response": output})
        print(f"--- Input {i} ---")
        print(f"Response: {output[:400]}")
        print()

### V1 Observations

**Q1: Does the output format vary across the 5 runs?**

TODO

**Q2: Does the model include information you didn't ask for?**

TODO

**Q3: Could you automatically parse this output (e.g., with `json.loads`)? Why or why not?**

TODO

**Q4: What would an aligned model do differently from a base model here? (Reference instruction tuning)**

TODO

---
## Version 2: The Improved Prompt (15 min)

**🔄 Switch driver/navigator.**

Apply the contract framework: add an **objective**, at least 2 **constraints**, and a **format specification**.

In [ ]:
# ============================================================
# VERSION 2: IMPROVED PROMPT (Contract Framework)
# ============================================================
# Apply at minimum:
#   - A clear OBJECTIVE statement
#   - At least 2 CONSTRAINTS
#   - A specified OUTPUT FORMAT
# ============================================================

improved_prompt = """
TODO: Write your improved prompt here.

Use the contract framework:
- OBJECTIVE: What should the model do?
- CONSTRAINTS: What should it avoid?
- OUTPUT FORMAT: What structure should the response have?
"""
# TODO: Replace with your V2 prompt

print(f"V2 Prompt:\n{improved_prompt}")

In [ ]:
# === Run V2 on the SAME 5 inputs ===
results_v2 = []

if OLLAMA_AVAILABLE:
    for i, inp in enumerate(test_inputs[:5]):
        prompt = improved_prompt.format(**inp)
        response = generate(prompt)
        results_v2.append({"input_id": i, "response": response})
        print(f"--- Input {i} ---")
        parsed, err = try_parse_json(response)
        if parsed:
            print(f"✓ Valid JSON: {json.dumps(parsed, indent=2)[:300]}")
        else:
            print(f"✗ Not valid JSON: {response[:300]}")
        print()
else:
    print("Using pre-computed V2 outputs:")
    v2_data = PRECOMPUTED.get(TASK_ID, {}).get("v2_improved", {})
    for i, output in enumerate(v2_data.get("outputs", [])[:5]):
        results_v2.append({"input_id": i, "response": output})
        parsed, err = try_parse_json(output)
        status = "✓ Valid JSON" if parsed else "✗ Not valid JSON"
        print(f"--- Input {i}: {status} ---")
        print(f"{output[:300]}\n")

### V2 Analysis

**Q1: Which issues from V1 are now resolved?**

TODO

**Q2: Is the output consistently parseable now?**

TODO

**Q3: Mechanistic reasoning — WHY did your changes help?** (Use terms: token distribution, attention, context, instruction tuning)

TODO

---
## Version 3: The Hardened Prompt (15 min)

Full contract: objective + constraints + format + evaluation criteria + **one few-shot example**.

In [ ]:
# ============================================================
# VERSION 3: HARDENED PROMPT (Full Contract + Example)
# ============================================================
# Include ALL FOUR contract components:
#   - OBJECTIVE
#   - CONSTRAINTS (including edge cases + adversarial handling)
#   - OUTPUT FORMAT
#   - EVALUATION CRITERIA
#   + At least ONE few-shot example
# ============================================================

hardened_prompt = """
TODO: Write your hardened prompt here.

Include:
1. OBJECTIVE
2. CONSTRAINTS (including empty input + adversarial input handling)
3. OUTPUT FORMAT (exact JSON structure)
4. EVALUATION CRITERIA
5. At least one EXAMPLE
"""
# TODO: Replace with your V3 prompt

print(f"V3 Prompt:\n{hardened_prompt}")

In [ ]:
# === Run V3 on ALL 7 inputs (5 normal + 1 edge + 1 adversarial) ===
results_v3 = []

if OLLAMA_AVAILABLE:
    for i, inp in enumerate(test_inputs):
        prompt = hardened_prompt.format(**inp)
        response = generate(prompt)
        results_v3.append({"input_id": i, "response": response})
        parsed, err = try_parse_json(response)
        label = "normal" if i < 5 else ("EDGE" if i == 5 else "ADVERSARIAL")
        status = "✓" if parsed else "✗"
        print(f"--- Input {i} [{label}] {status} ---")
        if parsed:
            print(json.dumps(parsed, indent=2)[:300])
        else:
            print(response[:300])
        print()
else:
    print("Using pre-computed V3 outputs:")
    v3_data = PRECOMPUTED.get(TASK_ID, {}).get("v3_hardened", {})
    for i, output in enumerate(v3_data.get("outputs", [])):
        results_v3.append({"input_id": i, "response": output})
        parsed, _ = try_parse_json(output)
        label = "normal" if i < 5 else ("EDGE" if i == 5 else "ADVERSARIAL")
        status = "✓" if parsed else "✗"
        print(f"--- Input {i} [{label}] {status} ---")
        print(f"{output[:300]}\n")

### V3 Analysis

**Q1: Compare V1 → V2 → V3 — which is most reliable?**

TODO

**Q2: How does V3 handle the edge case (empty input)?**

TODO

**Q3: How does V3 handle the adversarial case (prompt injection attempt)?**

TODO

**Q4: Full mechanistic justification — for each major change V1→V3, explain WHY it works using Modules 1–3:**

TODO

---
## Cross-Version Comparison Table (10 min)

Rate each metric: **Poor / Fair / Good / Excellent**

| Metric                  | V1 (Naive) | V2 (Improved) | V3 (Hardened) |
|-------------------------|------------|---------------|---------------|
| Format consistency      | TODO       | TODO          | TODO          |
| Constraint compliance   | TODO       | TODO          | TODO          |
| Edge case handling      | TODO       | TODO          | TODO          |
| Parseability (auto)     | TODO       | TODO          | TODO          |
| Hallucination frequency | TODO       | TODO          | TODO          |
| Overall quality         | TODO       | TODO          | TODO          |

**Key insight:** What is the single most impactful change from V1 → V3? Why did it work, mechanistically?

TODO

---
# PART 2 — BUILDING A TEST SUITE (20 minutes)

Create a test suite with **at least 10 test cases** across 3 categories:
- **Normal** (≥4): Standard inputs
- **Edge** (≥3): Empty fields, very long text, ambiguous input
- **Adversarial** (≥3): Prompt injection, misleading content, hallucination triggers

Build your test suite directly in `tests.json`.

In [ ]:
# === Load your test suite ===
# Edit tests.json with your 10+ test cases, then run this cell.

with open("tests.json", "r") as f:
    test_suite = json.load(f)

print(f"Loaded {len(test_suite)} test cases:")
for tc in test_suite:
    print(f"  [{tc['category']:12s}] {tc['id']}")

# Verify coverage
categories = {}
for tc in test_suite:
    cat = tc.get("category", "unknown")
    categories[cat] = categories.get(cat, 0) + 1

print(f"\nCoverage: {categories}")
assert len(test_suite) >= 10, f"Need at least 10 cases, got {len(test_suite)}"
assert categories.get('normal', 0) >= 4, f"Need ≥4 normal cases"
assert categories.get('edge', 0) >= 3, f"Need ≥3 edge cases"
assert categories.get('adversarial', 0) >= 3, f"Need ≥3 adversarial cases"
print("\n✓ Test suite meets minimum coverage requirements!")

In [ ]:
# === Run test suite against V3 (hardened prompt) ===

if OLLAMA_AVAILABLE:
    print("Running test suite against V3 hardened prompt...\n")
    report = run_test_suite(
        prompt_template=hardened_prompt,
        test_cases=test_suite,
        model="llama3.2:3b",
        temperature=0.3,
        verbose=True
    )
    
    print(f"\n{'=' * 60}")
    print(f"TEST RESULTS: {report['passed']}/{report['total']} passed ({report['pass_rate']:.0%})")
    print(f"Format compliance: {report['format_compliance_rate']:.0%}")
    print(f"Auto-checkable: {report['auto_passed']}/{report['auto_total']} passed")
    print(f"Manual review needed: {report['manual_review_count']}")
    print(f"{'=' * 60}")
    
    if report['failures']:
        print(f"\nFailures:")
        for f in report['failures']:
            print(f"  ✗ {f['test_id']} ({f['category']}): {f['reason']}")
else:
    print("⚠ Ollama unavailable. Test suite cannot be run automatically.")
    print("  Analyze the pre-computed V3 outputs above against your test cases manually.")
    print("  Document your manual evaluation below.")

### Test Suite Analysis

**Q1: What is your overall pass rate?**

TODO

**Q2: Which category has the most failures (normal/edge/adversarial)?**

TODO

**Q3: Are the failures due to format issues or content issues?**

TODO

**Q4: What would you change in V3 to address remaining failures?**

TODO

**Q5: Is 100% pass rate realistic? Why or why not?** (Hint: the model is probabilistic — Module 1)

TODO

---
# WRAP-UP

## Before You Submit

1. ✅ Notebook executed end-to-end
2. ✅ All TODO markers replaced
3. ✅ `prompt.md` filled in with all 3 versions + justifications + comparison table
4. ✅ `tests.json` has 10+ cases (≥4 normal, ≥3 edge, ≥3 adversarial)
5. ✅ Cross-version comparison table complete
6. ✅ `git add -A && git commit -m 'Lab 3 complete' && git push`

## Key Takeaways

1. TODO
2. TODO
3. TODO

---

## Portfolio Entries

Lab 3 contributes **two portfolio use cases**:
- **Use Case #2:** PR summarization (or your chosen task) — V1→V2→V3 with test suite
- **Use Case #3:** Action item extraction — if time permits, apply the contract framework to a second task from the task list as part of your homework

## Preview: Lab 4

Today you learned to write prompts as contracts. In **Lab 4 — Technique Toolbox**, you'll expand your toolkit: few-shot prompting in depth, chain-of-thought, self-consistency, and format constraints. Every technique will be grounded in the mechanisms studied so far and applied through the contract framework.

---

*Lab 3 of 8 — DevAssist / TaskFlow Lab Series*